In [1]:
import numpy as np
import pandas as pd
import transformers
import torch

from tqdm import tqdm, trange
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/CAN_Research/attack-free-1.csv', delimiter=',')
df['datetime'] = df['timestamp']
df['arbitration_id'] = df['arbitration_id'].astype(str).dropna().apply(lambda x: float.fromhex(x))
df['data_field'] = df['data_field'].astype(str).dropna().apply(lambda x: float.fromhex(x))
df['attack'] = 0
df = df.loc[:50000, ['datetime','arbitration_id', 'data_field', 'attack']]


df.head(5)

,datetime,arbitration_id,data_field,attack
0,1.672531e+09,193.0,3.458765e+18,0
1,1.672531e+09,197.0,3.458765e+18,0
2,1.672531e+09,388.0,8.589935e+09,0
3,1.672531e+09,455.0,1.790802e+15,0
4,1.672531e+09,461.0,0.000000e+00,0


In [4]:
dg = pd.read_csv('/content/drive/MyDrive/CAN_Research/DoS-1.csv', delimiter=',')
dg['datetime'] = dg['timestamp']
dg['arbitration_id'] = dg['arbitration_id'].astype(str).dropna().apply(lambda x: float.fromhex(x))
dg['data_field'] = dg['data_field'].astype(str).dropna().apply(lambda x: float.fromhex(x))
dg['attack'] = 1
dg = dg.loc[:50000, ['datetime','arbitration_id', 'data_field', 'attack']]

dg.head(5)

,datetime,arbitration_id,data_field,attack
0,1.672531e+09,485.0,5.044479e+18,1
1,1.672531e+09,489.0,4.503651e+15,1
2,1.672531e+09,249.0,1.225120e+17,1
3,1.672531e+09,761.0,6.313770e+11,1
4,1.672531e+09,409.0,1.498772e+19,1


In [5]:
# Feature and label extraction
from sklearn.utils import shuffle
data = pd.concat([df, dg])
data = shuffle(data)

X = data[['datetime','arbitration_id', 'data_field']]
y = data['attack']



In [6]:
data_columns = ['datetime','arbitration_id', 'data_field']
X['feature_string'] = data[data_columns].astype(str).apply(lambda x: '/'.join(x), axis=1)

In [7]:
# Prepare the final DataFrame
final_df = pd.DataFrame({'features': X['feature_string'], 'attack': y})

In [8]:
if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"

print(f'There are {torch.cuda.device_count()} GPU(s) available.')
print('Device name:', torch.cuda.get_device_name(0))

There are 1 GPU(s) available.
Device name: NVIDIA L4


In [9]:
# Count unique occurrences of 0s and 1s in the dataset
label_counts = final_df['attack'].value_counts()
print("Counts of unique labels:")
print(label_counts)

Counts of unique labels:
attack
1    50001
0    50001
Name: count, dtype: int64


In [10]:
sentences = final_df['features'].values
labels = final_df['attack'].values

In [11]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')
model = BertModel.from_pretrained('bert-large-uncased')

model.to(device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

In [12]:
# Tokenize the sentences in batches
batch_size = 32  # Adjust based on GPU memory
input_ids = []
attention_masks = []

def tokenize_batch(batch_sentences):
    batch_input_ids = []
    batch_attention_masks = []
    for sent in batch_sentences:
        encoded_dict = tokenizer.encode_plus(
                            sent,
                            add_special_tokens=True,
                            max_length=128,  # Adjust max_length based on your needs
                            padding='max_length',
                            return_attention_mask=True,
                            return_tensors='pt',
                       )
        batch_input_ids.append(encoded_dict['input_ids'])
        batch_attention_masks.append(encoded_dict['attention_mask'])
    return torch.cat(batch_input_ids, dim=0), torch.cat(batch_attention_masks, dim=0)

for i in range(0, len(sentences), batch_size):
    batch_sentences = sentences[i:i + batch_size]
    batch_input_ids, batch_attention_masks = tokenize_batch(batch_sentences)
    input_ids.append(batch_input_ids)
    attention_masks.append(batch_attention_masks)

input_ids = torch.cat(input_ids, dim=0).to(device)
attention_masks = torch.cat(attention_masks, dim=0).to(device)
labels = torch.tensor(labels).to(device)


def get_bert_embeddings(input_ids, attention_masks, model, batch_size):
    model.eval()  # Set the model to evaluation mode
    embeddings = []
    with torch.no_grad():
        for i in range(0, input_ids.size(0), batch_size):
            batch_input_ids = input_ids[i:i + batch_size]
            batch_attention_masks = attention_masks[i:i + batch_size]
            outputs = model(batch_input_ids, attention_mask=batch_attention_masks)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)  # Average pooling of the token embeddings
            embeddings.append(batch_embeddings)
    return torch.cat(embeddings, dim=0)

embeddings = get_bert_embeddings(input_ids, attention_masks, model, batch_size)


In [13]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), hidden_dim).to(device)
        c0 = torch.zeros(1, x.size(0), hidden_dim).to(device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

input_dim = embeddings.size(1)
hidden_dim = 128
output_dim = len(set(labels.cpu().numpy()))  # Number of classes
num_layers = 2

model_lstm = LSTMClassifier(input_dim, hidden_dim, output_dim, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

# Prepare DataLoader
dataset = TensorDataset(embeddings.unsqueeze(1), labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model_lstm(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

# Evaluation
X_test = embeddings.cpu().numpy()
y_test = labels.cpu().numpy()
model_lstm.eval()
with torch.no_grad():
    y_pred = model_lstm(embeddings.unsqueeze(1).to(device)).cpu().numpy()
y_pred = np.argmax(y_pred, axis=1)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


Epoch 1/10, Loss: 0.05562753975391388
Epoch 2/10, Loss: 0.6472563743591309
Epoch 3/10, Loss: 0.6569386720657349
Epoch 4/10, Loss: 0.6771913170814514
Epoch 5/10, Loss: 0.1658233255147934
Epoch 6/10, Loss: 0.002882890636101365
Epoch 7/10, Loss: 0.05611799657344818
Epoch 8/10, Loss: 0.04741519317030907
Epoch 9/10, Loss: 0.009921117685735226
Epoch 10/10, Loss: 0.19699986279010773
              precision    recall  f1-score   support

           0       0.93      0.80      0.86     50001
           1       0.83      0.94      0.88     50001

    accuracy                           0.87    100002
   macro avg       0.88      0.87      0.87    100002
weighted avg       0.88      0.87      0.87    100002

